**assignment 3 - NLPROC-ALV-B**

**Anja Kieburg, Friedemann Decker**

### Task 2 A

Korpus TIGER 2.2.doc (April 2018)
from: https://www.ims.uni-stuttgart.de/en/research/resources/corpora/tiger/

TIGER Corpus
The TIGER Corpus consists of approximate 900,000 tokens (50,000 sentences) of German newspaper text, taken from the Frankfurter Rundschau. The corpus was semi-automatically POS-tagged and annotated with syntactic structure. Moreover, it contains morphological and lemma information for terminal nodes

In [1]:
# Setup AI
%load_ext jupyter_ai_magics
%load_ext dotenv
%dotenv

ImportError: cannot import name 'PipelinePromptTemplate' from 'langchain_core.prompts' (C:\Users\Fried\AppData\Local\Programs\Python\Python311\Lib\site-packages\langchain_core\prompts\__init__.py)

In [2]:
import pandas as pd

**Step 1**: Import TIGER XML into Pandas, using only the entries under `s=Sätze`

In [3]:
tiger_corpus = pd.read_xml("tiger_release_aug07.corrected.16012013.xml",
                  xpath="//s/graph/terminals/t",
                  elems_only=False,
                  encoding="ISO-8859-1")

In [5]:
tiger_corpus.head()

,id,word,lemma,pos,morph,case,number,gender,person,degree,tense,mood,secedge
0,s1_1,``,--,$(,--,--,--,--,--,--,--,--,NaN
1,s1_2,Ross,Ross,NE,Nom.Sg.Masc,Nom,Sg,Masc,--,--,--,--,NaN
2,s1_3,Perot,Perot,NE,Nom.Sg.Masc,Nom,Sg,Masc,--,--,--,--,NaN
3,s1_4,wäre,sein,VAFIN,3.Sg.Past.Subj,--,Sg,--,3,--,Past,Subj,NaN
4,s1_5,vielleicht,vielleicht,ADV,--,--,--,--,--,--,--,--,NaN


**Step 2**: Required fields: Import "word" and "pos" into the DataFrame

In [66]:
words = tiger_corpus[["word", "pos"]]
words.head()

,word,pos
0,``,$(
1,Ross,NE
2,Perot,NE
3,wäre,VAFIN
4,vielleicht,ADV


**Step 3**: Count the number of unique combinations of "word" AND "pos" = frequency of occurrence in the corpus

In [67]:
word_pos_freq = words[["word", "pos"]].value_counts()
words = word_pos_freq.reset_index(name="pos_freq")
words.head()

,word,pos,pos_freq
0,",","$,",43656
1,.,$.,41057
2,der,ART,25286
3,die,ART,21209
4,und,KON,16281


**Step 4**: Convert "word" to lowercase so that different parts of speech (POS) can be grouped into a "word_lower" category

In [68]:
word_lower = words["word"].str.lower()
words.insert(1, "word_lower", word_lower)
words.head()

,word,word_lower,pos,pos_freq
0,",",",","$,",43656
1,.,.,$.,41057
2,der,der,ART,25286
3,die,die,ART,21209
4,und,und,KON,16281


**Step 5**: Determine the number of words per "word_lower" category and enter it as "word_freq" in each row to calculate p(t|w).

In [69]:
word_freq = words.groupby("word_lower")["pos_freq"].transform("sum")
words.insert(4, "word_freq", word_freq)
words.head()

,word,word_lower,pos,pos_freq,word_freq
0,",",",","$,",43656,43659
1,.,.,$.,41057,41057
2,der,der,ART,25286,29662
3,die,die,ART,21209,30024
4,und,und,KON,16281,16786


**Step 6**: Calculate p_pos = p(t|w)

In [70]:
p_pos = words["pos_freq"] / words["word_freq"]
words.insert(5, "p_pos", p_pos)
words.head()

,word,word_lower,pos,pos_freq,word_freq,p_pos
0,",",",","$,",43656,43659,0.999931
1,.,.,$.,41057,41057,1.000000
2,der,der,ART,25286,29662,0.852471
3,die,die,ART,21209,30024,0.706402
4,und,und,KON,16281,16786,0.969915


**Step 7**: Keep only p_pos_max values in the DataFrame

In [71]:
words["max_p_pos"] = words.groupby("word_lower")["p_pos"].transform("max")
words = words[words["p_pos"] == words["max_p_pos"]]

words.head()

,word,word_lower,pos,pos_freq,word_freq,p_pos,max_p_pos
0,",",",","$,",43656,43659,0.999931,0.999931
1,.,.,$.,41057,41057,1.000000,1.000000
2,der,der,ART,25286,29662,0.852471,0.852471
3,die,die,ART,21209,30024,0.706402,0.706402
4,und,und,KON,16281,16786,0.969915,0.969915


**Step 8**: Delete columns that aren't needed (clean up). Rename the columns according to the assignment.

In [72]:
words = words.drop(columns = ["word_lower", "p_pos"])


In [78]:
words = words.rename(columns={'word': 'w', 'pos': 't', 'max_p_pos': 'p'})

**Result**: Computation of the tag **t** for each word **w** that maximizes **p(t|w)**.
List of the first 30 values:

In [79]:
words.head(30)

,w,t,pos_freq,word_freq,p
0,",","$,",43656,43659,0.999931
1,.,$.,41057,41057,1.000000
2,der,ART,25286,29662,0.852471
3,die,ART,21209,30024,0.706402
4,und,KON,16281,16786,0.969915
5,in,APPR,13711,15102,0.907893
6,den,ART,9486,9955,0.952888
7,'',$(,8811,8811,1.000000
8,``,$(,8809,8809,1.000000
9,von,APPR,8182,9033,0.905790


**Appendix**: Meaning of the Abbreviations Used for Parts of Speech (POS)

In [80]:
tiger_corpus_pos_abr = pd.read_xml("tiger_release_aug07.corrected.16012013.xml",
                  xpath="//head/annotation/feature/value",
                  elems_only=False,
                  encoding="ISO-8859-1")

In [81]:
tiger_corpus_pos_abr.head(56)

,name,value
0,$(,Sonstige Satzzeichen; satzintern
1,"$,",Komma
2,$.,Satzbeendende Interpunktion
3,--,<Nicht zugeordnet>
4,ADJA,Attributives Adjektiv
5,ADJD,Adverbiales oder prädikatives Adjektiv
6,ADV,Adverb
7,APPO,Postposition
8,APPR,Präposition; Zirkumposition links
9,APPRART,Präposition mit Artikel


### Task 2 B - HMM tagger

a) Split the corpus into a training set and test set.

In [7]:
%%ai gemini:gemini-1.5-pro
Welche Splitting-Ratio empfiehlst du für einen Datensatz mit 900.000 Einträgen?

Cannot determine model provider from model ID `gemini:gemini-1.5-pro`.

To see a list of models you can use, run `%ai list`

If you were trying to run a command, run `%ai help` to see a list of commands.

In [1]:
%ai list

UsageError: Line magic function `%ai` not found.


In [11]:
%pip install langchain-google-genai

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.3.30 requires langchain-core<1.0.0,>=0.3.85, but you have langchain-core 1.7.6 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached langchain_core-1.7.6-py3-none-any.whl.metadata (4.8 kB)
Using cached langchain_core-1.7.6-py3-none-any.whl (580 kB)
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.86
    Uninstalling langchain-core-0.3.86:
      Successfully uninstalled langchain-core-0.3.86


Training Set = 95% und Test Set = 5% aller Sätze des Korpus. Bei ca. 900.000 Datensätzen ergeben das ca. 45.000 Test-Datensätze, die genügen um statistisch verlässliche Testergebnisse zu liefern.

b) From the training set, count how often each POS tag occurs,
how often one POS tag follows another and how often each word occurs with each POS tag.

c) Compute the transition and observation probabilities of the HMM tagger from this.

d) Then implement the Viterbi algorithm so you can decode a test sentence.

e) Create a Viterbi table
that stores, for each position in the sentence and each possible tag, the probability of the most
probable path ending in that tag.

f) Initialize the table for the first word using the initial tag probabilities and the observation probability of that word.

g) Process the remaining words from left to right.

h) For each word, update the Viterbi table by considering all possible tags and the probabilities of the most probable paths leading to them.

i) Store both the best score and the preceding tag that achieved it (the backpointer).

j) After filling the table, backtrack from the last word to find the most probable sequence of tags.

k) Now run your algorithm on the test set. Report its error rate and compare its performance
to the most frequent tag baseline.